# EDA — Eventos AWS CloudTrail

**Análise Exploratória de Dados** sobre 1000 eventos sintéticos simulando logs do AWS CloudTrail.

Como engenheiro de infraestrutura, esse tipo de análise é o dia a dia de quem investiga:
- Padrões de uso de APIs da AWS
- Anomalias de segurança (ex.: rajadas de `AccessDenied`)
- Janelas de pico de atividade

**Perguntas respondidas:**
1. Quais as 10 chamadas de API mais frequentes?
2. Quais horas do dia têm mais atividade?
3. Qual o percentual de eventos com erro? E por tipo de erro?
4. Quais IPs de origem têm mais erros `AccessDenied`?
5. Existe correlação entre hora do dia e taxa de erro?

> Antes de rodar este notebook, gere o dataset com `make data` na raiz do projeto.

In [ ]:
# Imports e configuração do tema escuro
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.style.use("dark_background")
COR_PRINCIPAL = "#4FC3F7"
COR_ALERTA = "#EF5350"

# Carrega o dataset e cria colunas derivadas de tempo
CSV = Path("..") / "data" / "cloudtrail_sample.csv"
df = pd.read_csv(CSV, parse_dates=["timestamp"])
df["hora"] = df["timestamp"].dt.hour
df["dia_semana"] = df["timestamp"].dt.dayofweek

print(f"{len(df)} eventos | {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

## Q1 — Quais as 10 chamadas de API mais frequentes?

Em contas reais, chamadas de **leitura** (`Describe*`) costumam dominar o volume — 
são feitas por ferramentas de monitoramento, Terraform plans e pelo console. 
Chamadas **destrutivas** (`Terminate*`, `Delete*`) devem ser raras; se aparecerem no topo, é sinal de alerta.

In [ ]:
# Contagem das 10 APIs mais chamadas
top10 = df["event_name"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
top10.sort_values().plot.barh(ax=ax, color=COR_PRINCIPAL)
ax.set_title("Q1 — Top 10 chamadas de API mais frequentes")
ax.set_xlabel("Número de eventos")
plt.tight_layout()
plt.show()

top10

## Q2 — Quais horas do dia têm mais atividade?

A distribuição por hora revela o **perfil de uso** da conta: um pico em horário comercial 
indica atividade humana; atividade relevante de madrugada indica automação (cron, pipelines) 
— ou algo que não deveria estar lá.

In [ ]:
# Eventos agregados por hora do dia (0-23)
por_hora = df.groupby("hora").size().reindex(range(24), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
por_hora.plot.line(ax=ax, marker="o", color=COR_PRINCIPAL)
ax.set_title("Q2 — Atividade por hora do dia")
ax.set_xlabel("Hora (UTC)")
ax.set_ylabel("Número de eventos")
ax.set_xticks(range(0, 24, 2))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Hora de pico: {por_hora.idxmax()}h com {por_hora.max()} eventos")

## Q3 — Qual o percentual de eventos com erro? E por tipo?

No CloudTrail, a ausência de `errorCode` significa sucesso. A **taxa de erro global** é um 
indicador de saúde da conta, e a quebra por tipo aponta a causa: `AccessDenied` sugere 
problema de IAM (ou ataque), `NoSuchBucket` sugere bug de aplicação.

In [ ]:
# Taxa de erro global e contagem por tipo
taxa_total = df["error_code"].notna().mean() * 100
por_tipo = df["error_code"].value_counts(dropna=True)
pct_por_tipo = (por_tipo / len(df) * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
por_tipo.plot.bar(ax=ax, color=[COR_ALERTA, "#FFB74D", "#BA68C8"])
ax.set_title(f"Q3 — Erros por tipo (taxa global: {taxa_total:.1f}%)")
ax.set_ylabel("Número de eventos")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

print(f"Taxa de erro global: {taxa_total:.1f}%")
print("\nPercentual por tipo (sobre o total de eventos):")
print(pct_por_tipo.to_string())

## Q4 — Quais IPs de origem têm mais erros `AccessDenied`?

Esta é a análise mais próxima de **detecção de ameaças**: um IP concentrando muitos 
`AccessDenied` pode indicar credencial vazada sendo testada, scanner automatizado, 
ou uma role mal configurada em loop de retry.

In [ ]:
# Top 10 IPs com mais AccessDenied
negados = df[df["error_code"] == "AccessDenied"]
top_ips = negados["source_ip"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
top_ips.sort_values().plot.barh(ax=ax, color=COR_ALERTA)
ax.set_title("Q4 — Top 10 IPs com mais AccessDenied")
ax.set_xlabel("Número de AccessDenied")
plt.tight_layout()
plt.show()

top_ips

## Q5 — Existe correlação entre hora do dia e taxa de erro?

O heatmap **hora × dia da semana** mostra *quando* os erros se concentram. 
Se a taxa de erro da madrugada for muito maior que a do horário comercial, 
os erros não acompanham o volume de uso — ou seja, há uma fonte de falhas 
independente da atividade normal (automação quebrada ou atividade maliciosa).

In [ ]:
# Taxa de erro (%) por dia da semana x hora do dia
DIAS = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
pivot = (
    df.assign(tem_erro=df["error_code"].notna())
    .pivot_table(index="dia_semana", columns="hora", values="tem_erro", aggfunc="mean")
    .reindex(index=range(7), columns=range(24))
    * 100
)
pivot.index = DIAS

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, ax=ax, cmap="rocket", cbar_kws={"label": "Taxa de erro (%)"},
            linewidths=0.3, linecolor="#222222")
ax.set_title("Q5 — Taxa de erro (%) por hora x dia da semana")
ax.set_xlabel("Hora (UTC)")
plt.tight_layout()
plt.show()

# Comparação direta: madrugada vs horário comercial
madrugada = df[df["hora"].between(2, 5)]["error_code"].notna().mean() * 100
comercial = df[df["hora"].between(9, 18)]["error_code"].notna().mean() * 100
print(f"Taxa de erro na madrugada (2h-5h): {madrugada:.1f}%")
print(f"Taxa de erro em horário comercial (9h-18h): {comercial:.1f}%")

## Conclusões

1. **Leituras dominam**: as APIs `Describe*` lideram o volume, como esperado em contas reais.
2. **Pico comercial**: a atividade se concentra em horário de trabalho, com pico à tarde.
3. **Anomalia de segurança**: poucos IPs concentram a maioria dos `AccessDenied`, e a taxa de erro 
   da madrugada é muito superior à do horário comercial — padrão típico de credencial comprometida 
   sendo testada fora do horário de monitoramento.

**Próximo passo em um cenário real**: criar um alarme (CloudWatch/EventBridge) para rajadas de 
`AccessDenied` por IP e revogar as credenciais associadas aos IPs suspeitos.